# Download data

TITAN slide embeddings (Hugging Face) + TCGA RNA-seq and clinical (GDC), then package into `artifacts/data/processed`.

Run from the repo root. Every download cell is resumable: rerun it and finished files are skipped.

In [ ]:
%pip install -q pandas pyarrow requests tqdm huggingface_hub ipywidgets

In [ ]:
import sys
sys.path.insert(0, "src")

import config as cfg
import download as dl

MANIFEST = cfg.MANIFEST_DIR / dl.MANIFEST_NAME

## 1. Hugging Face login

Paste a token from https://huggingface.co/settings/tokens. The account must have been granted access on the `MahmoodLab/TITAN` model page. Skip this cell if `HF_TOKEN` is set or you are already logged in.

In [ ]:
from huggingface_hub import login

login()

## 2. TITAN embeddings (one file)

In [ ]:
dl.download_titan(cfg.RAW_DIR)

## 3. GDC clinical + RNA manifest (metadata only, fast)

In [ ]:
dl.download_clinical(cfg.PROJECTS, cfg.RAW_DIR)
files = dl.build_rna_manifest(cfg.PROJECTS, cfg.MANIFEST_DIR)
files.groupby(["project_id", "sample_type"]).size()

## 4. GDC RNA download

Interrupt any time; rerun the cell to resume.

In [ ]:
dl.download_rna(MANIFEST, cfg.RAW_DIR, workers=8)

## 5. Status

In [ ]:
dl.rna_status(MANIFEST, cfg.RAW_DIR)
print("titan:", "present" if (cfg.RAW_DIR / "titan" / cfg.TITAN_FILE).exists() else "missing")
print("clinical:", "present" if (cfg.RAW_DIR / "clinical.json").exists() else "missing")

## 6. Package

In [ ]:
cfg.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clinical = dl.package_clinical(cfg.RAW_DIR, cfg.PROCESSED_DIR)
dl.package_titan(cfg.RAW_DIR, cfg.PROCESSED_DIR, clinical)
dl.package_rna(cfg.RAW_DIR, cfg.MANIFEST_DIR, cfg.PROCESSED_DIR)